In [1]:
import pandas as pd
import sqlite3

In [2]:
import os
os.getcwd() # get current working directory

'c:\\Users\\ngo1\\Documents\\2026-10-BILBAO-FT-Data-Science\\2-Data_Analysis\\8-BBDD\\SQL\\Teoria\\SQL_Python\\ejercicio'

In [3]:
# Conectamos con la base de datos
connection = sqlite3.connect("cervezas.db")

# Obtenemos un cursor que utilizaremos para hacer las queries
crsr = connection.cursor()

In [4]:
crsr.execute("PRAGMA foreign_keys = ON;") # Activamos las claves foráneas

In [5]:
query = '''
 CREATE TABLE Cervezas (
     CodC varchar(255) PRIMARY KEY,
     Envase varchar(255),
     Capacidad float,
     Stock int
 );
 '''
crsr.execute(query)

In [6]:
query = '''
 CREATE TABLE Bares (
     CodB varchar(255) PRIMARY KEY,
     Cif varchar(255),
     Nombre varchar(255),
     Localidad varchar(255)
 );
 '''
crsr.execute(query)

In [7]:
query = '''
 CREATE TABLE Empleados (
     CodE varchar(255) PRIMARY KEY,
     Nombre varchar(255),
     Sueldo int
 );
 '''
crsr.execute(query)

In [8]:
query = '''
 CREATE TABLE Reparto (
     CodE varchar(255), 
     CodB varchar(255),
     CodC varchar(255),
     Fecha text,
     Cantidad int,
     Foreign Key (CodE) REFERENCES Empleados(CodE),
     Foreign Key (CodB) REFERENCES Bares(CodB),
     Foreign Key (CodC) REFERENCES Cervezas(CodC)
 );
 '''
crsr.execute(query)

In [9]:
query = '''
 INSERT INTO Cervezas (CodC, Envase, Capacidad, Stock)
 VALUES 
    ('01', 'Botella', 0.2, 3600),
    ('02', 'Botella', 0.33, 1200),
    ('03', 'Lata', 0.33, 2400),
    ('04', 'Botella', 1.0, 288),
    ('05', 'Barril', 60.0, 30);
'''
crsr.execute(query)

In [10]:
query = '''
 INSERT INTO Bares (CodB, Cif, Nombre, Localidad)
 VALUES 
    ('001', '11111111X', 'Stop', 'Villa Botijo'),
    ('002', '22222222Y', 'Las Vegas', 'Villa Botijo'),
    ('003', '-', 'Club Social', 'Las Ranas'),
    ('004', '33333333Z', 'Otra Ronda', 'La Esponja');
'''
crsr.execute(query)

In [11]:
query = '''
 INSERT INTO Empleados (CodE, Nombre, Sueldo)
 VALUES 
    ('1', 'Prudencio Caminero', 120000),
    ('2', 'Vicente Merario', 110000),
    ('3', 'Valentin Siempre', 100000);
'''
crsr.execute(query)

In [12]:
query = '''
INSERT INTO Reparto (CodE, CodB, CodC, Fecha, Cantidad) 
VALUES
('1', '001', '01', '2005-10-21', 240),
('1', '001', '02', '2005-10-21', 48),
('1', '002', '03', '2005-10-22', 60),
('1', '004', '05', '2005-10-22', 4),
('2', '002', '03', '2005-10-22', 48),
('2', '002', '05', '2005-10-23', 2),
('2', '004', '01', '2005-10-23', 480),
('2', '004', '02', '2005-10-24', 72),
('3', '003', '03', '2005-10-24', 48),
('3', '003', '04', '2005-10-25', 20);
'''
crsr.execute(query)

In [13]:
query = '''
 SELECT * FROM Cervezas
 '''
crsr.execute(query).fetchall()

[('01', 'Botella', 0.2, 3600),
 ('02', 'Botella', 0.33, 1200),
 ('03', 'Lata', 0.33, 2400),
 ('04', 'Botella', 1.0, 288),
 ('05', 'Barril', 60.0, 30)]

In [14]:
connection.commit()

# EJERCICIOS

In [ ]:
# 1. Obtener el nombre de los empleados que hayan repartido al bar Stop durante la semana del 17 al 23 de octubre de 2005.

query = '''
SELECT DISTINCT Empleados.Nombre
FROM Empleados
JOIN Reparto ON Reparto.CodE = Empleados.CodE
JOIN Bares ON Bares.CodB = Reparto.CodB
WHERE Bares.Nombre = "Stop" AND Reparto.Fecha BETWEEN '2005-10-17' AND '2005-10-23';
'''

crsr.execute(query).fetchall()

[('Prudencio Caminero',)]

In [ ]:
# 1. Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo Botella y capacidad inferior a 1 litro, ordenados por localidad.

query = '''
SELECT DISTINCT Bares.Nombre, Bares.Cif
FROM Bares
JOIN Reparto ON Bares.CodB = Reparto.CodB
JOIN Cervezas ON Cervezas.CodC = Reparto.CodC
WHERE Cervezas.Envase = "Botella" AND Cervezas.Capacidad < 1
ORDER BY Bares.Localidad;
'''

crsr.execute(query).fetchall()

[('Otra Ronda', '33333333Z'), ('Stop', '11111111X')]

In [19]:
pd.read_sql_query(query, connection)

,Nombre,Cif
0,Otra Ronda,33333333Z
1,Stop,11111111X


In [21]:
# 3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad) realizados por Prudencio Caminero.

query = '''
SELECT Bares.Nombre, Cervezas.Envase, Cervezas.Capacidad, Reparto.Fecha, Reparto.Cantidad
FROM Cervezas
JOIN Reparto ON Reparto.CodC = Cervezas.CodC
JOIN Bares ON Bares.CodB = Reparto.CodB
JOIN Empleados ON Empleados.CodE = Reparto.CodE
WHERE Empleados.Nombre = "Prudencio Caminero";
'''

pd.read_sql_query(query, connection)

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,2005-10-21,240
1,Stop,Botella,0.33,2005-10-21,48
2,Las Vegas,Lata,0.33,2005-10-22,60
3,Otra Ronda,Barril,60.00,2005-10-22,4


In [28]:
# 4. Obtener los bares a los que se les ha repartido envases de tipo botella y capacidad 0.2 ó 0.33.

query = '''
SELECT DISTINCT Bares.Nombre AS Bares
FROM Bares
JOIN Reparto ON Reparto.CodB = Bares.CodB
JOIN Cervezas ON Cervezas.CodC = Reparto.CodC
WHERE Cervezas.Envase = "Botella" AND (Cervezas.Capacidad = 0.2 OR Cervezas.Capacidad = 0.33);
'''

pd.read_sql_query(query, connection)

,Bares
0,Stop
1,Otra Ronda


In [29]:
# 5. Nombre de los empleados que han repartido a los bares "Stop" y "Las Vegas" cervezas con envase botella.

query = '''
SELECT DISTINCT Empleados.Nombre
FROM Empleados
JOIN Reparto ON Reparto.CodE = Empleados.CodE
JOIN Cervezas ON Cervezas.CodC = Reparto.CodC
JOIN Bares ON Bares.CodB = Reparto.CodB
WHERE (Bares.Nombre = "Stop" OR Bares.Nombre = "Las Vegas") AND Cervezas.Envase = "Botella";
'''

pd.read_sql_query(query, connection)

,Nombre
0,Prudencio Caminero


In [30]:
# 6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de Villa Botijo.

query = '''
SELECT Empleados.Nombre, COUNT(Reparto.Fecha) AS "Nº de viajes"
FROM Empleados
JOIN Reparto ON Reparto.CodE = Empleados.CodE
JOIN Bares ON Bares.CodB = Reparto.CodB
WHERE Bares.Localidad <> "Villa Botijo"
GROUP BY Reparto.CodE;
'''

pd.read_sql_query(query, connection)

,Nombre,Nº de viajes
0,Prudencio Caminero,1
1,Vicente Merario,2
2,Valentin Siempre,2


In [32]:
# 7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.

query = '''
SELECT Bares.Nombre, Bares.Localidad, SUM(Cervezas.Capacidad * Reparto.Cantidad) AS "Litros comprados"
FROM Bares
JOIN Reparto ON Reparto.CodB = Bares.CodB
JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
GROUP BY Bares.CodB
ORDER BY "Litros comprados" DESC
LIMIT 1;
'''

pd.read_sql_query(query, connection)

,Nombre,Localidad,Litros comprados
0,Otra Ronda,La Esponja,359.76


In [33]:
# 8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y capacidad menor que 1 litro.

query = '''
SELECT DISTINCT Bares.Nombre, Bares.Localidad
FROM Bares
JOIN Reparto ON Reparto.CodB = Bares.CodB
JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
WHERE Cervezas.Envase = "Botella" AND Cervezas.Capacidad < 1;
'''

pd.read_sql_query(query, connection)

,Nombre,Localidad
0,Stop,Villa Botijo
1,Otra Ronda,La Esponja


In [36]:
# 9. Subir un 5% el sueldo del empleado que más días haya trabajado.

query = '''
SELECT Empleados.Nombre, Empleados.Sueldo, COUNT(DISTINCT Reparto.Fecha) AS "Días Trabajados"
FROM Empleados
JOIN Reparto ON Reparto.CodE = Empleados.CodE
GROUP BY Empleados.CodE
ORDER BY "Días Trabajados" DESC
LIMIT 1
;
'''

pd.read_sql_query(query, connection)

,Nombre,Sueldo,Días Trabajados
0,Vicente Merario,110000,3


In [41]:
query = '''
UPDATE Empleados
SET Sueldo = Sueldo * 1.05
WHERE CodE = (
    SELECT CodE
    FROM Reparto
    GROUP BY CodE
    ORDER BY COUNT(DISTINCT Fecha) DESC
    LIMIT 1
);
'''

crsr.execute(query)

In [42]:
query = '''
SELECT *
FROM Empleados
;
'''

pd.read_sql_query(query, connection)

,CodE,Nombre,Sueldo
0,1,Prudencio Caminero,120000
1,2,Vicente Merario,115500
2,3,Valentin Siempre,100000


In [43]:
# 10. Insertar un nuevo reparto del empleado "Vicente Merario" al bar "Stop" de 48 cervezas de tipo lata el día 10/26/05.

query = '''
INSERT INTO Reparto (CodE, CodB, CodC, Fecha, Cantidad) 
VALUES (
(SELECT CodE FROM Empleados WHERE Empleados.Nombre = "Vicente Merario"),
(SELECT CodB FROM Bares WHERE Bares.Nombre = "Stop"),
(SELECT CodC FROM Cervezas WHERE Cervezas.Envase = "Lata"),
'2005-10-26',
48
);
'''

crsr.execute(query)

In [44]:
query = '''
SELECT *
FROM Reparto
;
'''

pd.read_sql_query(query, connection)

,CodE,CodB,CodC,Fecha,Cantidad
0,1,001,01,2005-10-21,240
1,1,001,02,2005-10-21,48
2,1,002,03,2005-10-22,60
3,1,004,05,2005-10-22,4
4,2,002,03,2005-10-22,48
5,2,002,05,2005-10-23,2
6,2,004,01,2005-10-23,480
7,2,004,02,2005-10-24,72
8,3,003,03,2005-10-24,48
9,3,003,04,2005-10-25,20
